# PPO Bootstrap Curriculum

Warm-start a `MaskablePPO` policy by walking it through a fixed curriculum of
`(map, opponent)` stages before handing the resulting checkpoint off to
self-play (`notebooks/ppo_training.ipynb`).

Six stages: starter map (random → simple → medium), then beginner map
(random → simple → medium). A stage advances only when its eval win-rate
stays at or above its `promotion_win_rate` for `patience` consecutive
evaluations. If a stage exhausts its `max_timesteps` budget without
promoting, the runner raises `CurriculumStalled` — fix the underlying
issue (reward shaping, hyperparams) before raising the budget.

Stage list lives in `configs/bootstrap.yaml`; edit there to tune
thresholds, budgets, or PPO hyperparams without touching this notebook.

In [ ]:
# 1. Imports.
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.bootstrap import (
    CurriculumStalled,
    load_bootstrap_config,
    run_curriculum,
)
from reinforcetactics.rl.evaluation import evaluate_model
from reinforcetactics.rl.masking import make_maskable_env

## 2. Configuration

Reads `configs/bootstrap.yaml`. To experiment, copy that file, edit, and
point `CONFIG_PATH` at the copy.

In [ ]:
CONFIG_PATH = Path('configs/bootstrap.yaml')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('benchmarks') / 'bootstrap' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_bootstrap_config(CONFIG_PATH)

print(f'Config:        {CONFIG_PATH}')
print(f'Output dir:    {OUTPUT_DIR}')
print(f'Stages:        {len(cfg.stages)}')
for s in cfg.stages:
    print(
        f"  - {s.name:18s}  map={s.map_file:25s}  opp={s.opponent:6s}  "
        f"WR>={s.promotion_win_rate:.0%}  budget={s.max_timesteps:>10,}"
    )
print(f'Eval freq:     {cfg.eval_freq:,} env steps')
print(f'n_envs:        {cfg.n_envs}')
print(f'PPO ent_coef:  {cfg.ppo.ent_coef}')
print(f'Total budget:  {sum(s.max_timesteps for s in cfg.stages):,} env steps (worst case)')

## 3. Run the curriculum

One `model.learn()` call per stage, with the env swapped between stages
via `model.set_env(...)`. The shared `MaskablePPO` instance survives the
whole run, so the policy carries forward; only the `(map, opponent)` pair
changes. Best-by-eval-win-rate checkpoints land in
`OUTPUT_DIR/<stage_name>/best_model.zip`; the post-promotion model is
written as `stage_final.zip`.

In [ ]:
try:
    result = run_curriculum(cfg, output_dir=OUTPUT_DIR)
    stalled = None
except CurriculumStalled as exc:
    print(f'\nSTALLED: {exc}')
    print('Investigate before raising the budget \u2014 a stuck stage usually')
    print('signals reward shaping or hyperparam issues, not insufficient steps.')
    result = None
    stalled = exc

if result is not None:
    print(f"\nFinal model: {result['final_model_path']}")
    for h in result['history']:
        last = h['results'][-1] if h['results'] else None
        last_wr = f"{last['win_rate']:.1%}" if last else 'n/a'
        print(
            f"  {h['stage']:18s}  promoted={h['promoted']!s:5s}  "
            f"best_wr={h['best_win_rate']:.1%}  last_wr={last_wr}"
        )

## 4. Win-rate over the full curriculum

Concatenates eval snapshots across stages onto a single timestep axis.
Vertical lines mark stage transitions; horizontal lines mark each stage's
promotion threshold. A regression bump right after a transition is
expected with hard cutoffs — the policy is encountering a new opponent
for the first time.

In [ ]:
if result is None:
    print('Skipped \u2014 no successful run to plot.')
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.get_cmap('tab10')
    for i, h in enumerate(result['history']):
        if not h['results']:
            continue
        xs = [r['timesteps'] for r in h['results']]
        ys = [r['win_rate'] for r in h['results']]
        ax.plot(xs, ys, marker='o', label=h['stage'], color=cmap(i % 10))
        # Promotion threshold for the stage.
        stage_cfg = next(s for s in cfg.stages if s.name == h['stage'])
        ax.hlines(
            stage_cfg.promotion_win_rate,
            xmin=xs[0],
            xmax=xs[-1],
            colors=cmap(i % 10),
            linestyles=':',
            alpha=0.5,
        )
        # Stage transition marker at the last timestep of the stage.
        ax.axvline(xs[-1], color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Cumulative env timesteps')
    ax.set_ylabel('Eval win rate')
    ax.set_ylim(-0.02, 1.02)
    ax.set_title('Curriculum win rate (dotted = stage threshold, dashed = transition)')
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=9)
    out = OUTPUT_DIR / 'curriculum_winrate.png'
    fig.tight_layout()
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 5. Sanity eval on the final stage

Reload the saved checkpoint and run a fresh evaluation against the
hardest stage (beginner map vs medium bot) to confirm the exit criterion
really holds. Uses more episodes than the in-training eval for a tighter
estimate.

In [ ]:
if result is None:
    print('Skipped \u2014 no final checkpoint.')
else:
    final_stage = cfg.stages[-1]
    sanity_env = make_maskable_env(
        map_file=final_stage.map_file,
        opponent=final_stage.opponent,
        max_steps=cfg.env.max_steps,
        max_turns=cfg.env.max_turns,
        reward_config=cfg.env.reward_config,
        enabled_units=cfg.env.enabled_units,
        action_space_type=cfg.env.action_space_type,
        seed=cfg.seed + 9999,  # different seed than training eval
    )
    loaded = MaskablePPO.load(result['final_model_path'])
    metrics = evaluate_model(loaded, sanity_env, n_episodes=100, seed=cfg.seed + 9999)
    sanity_env.close()
    print(
        f"Sanity eval ({final_stage.name}, n=100):  "
        f"WR={metrics['win_rate']:.1%}  "
        f"reward={metrics['avg_reward']:+.1f}  "
        f"W/L/D={metrics['wins']}/{metrics['losses']}/{metrics['draws']}"
    )
    if metrics['win_rate'] < final_stage.promotion_win_rate:
        print(
            f"  WARNING: sanity WR {metrics['win_rate']:.1%} < threshold "
            f"{final_stage.promotion_win_rate:.1%}. The policy may be\n"
            "  overfit to the training eval seed. Consider a longer final\n"
            "  stage or rolling-window promotion before warm-starting self-play."
        )

## 6. Hand off to self-play

The final checkpoint is the artifact `notebooks/ppo_training.ipynb`'s
self-play setup should load. Drop this snippet into that notebook in
place of the fresh `MaskablePPO(...)` constructor call:

```python
from sb3_contrib import MaskablePPO

BOOTSTRAP_CHECKPOINT = '<paste path printed below>'
model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)
```

`set_env()` is called implicitly by `MaskablePPO.load(..., env=...)`, so
the policy weights load while the env is the new self-play one.

In [ ]:
if result is not None:
    print('Hand-off path:')
    print(f"  {result['final_model_path']}")
    print('\nIn ppo_training.ipynb:')
    print(f"  BOOTSTRAP_CHECKPOINT = '{result['final_model_path']}'")
    print('  model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)')